In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Ajuste o caminho conforme sua estrutura
sys.path.append(str(Path("..").resolve()))

from service.pns_service import get_dataframe, register_derived_variable, list_variables

In [3]:
df_variables = list_variables()
print(df_variables.to_string())


                     variable                                                                                                                                                                                                                                                                                                      descricao categoria tipo_dado                                                           regra_derivacao                                           depends_on  code_2013 type_2013  exists_2013  code_2019 type_2019  exists_2019
0        anos_estudo_estimado                                                                                                                                                                                                                                                                  Anos de estudo (proxy via escolaridade_nivel)  derivada     float                                 Calculada a partir de: escolaridade_nivel                          

In [ ]:
# ==============================================================================
# 1. FUNÇÕES AUXILIARES DE LÓGICA
# ==============================================================================

def calcular_anos_estudo_proxy(df):
    mapping = {'1': 0, '2': 4, '3': 8, '4': 11, '5': 11, '6': 15, '7': 16}
    return df['escolaridade_nivel'].astype(str).str.replace(r'\.0$', '', regex=True).map(mapping).fillna(0)

def classificar_cobertura_explicita(df, col_sus, col_pago, col_plano=None):
    """
    Versão explicita que aceita os nomes exatos das colunas para evitar erros de digitação.
    """
    # Normalização segura
    is_sus = df[col_sus].astype(str).str.replace(r'\.0$', '', regex=True) == '1'
    is_pago = df[col_pago].astype(str).str.replace(r'\.0$', '', regex=True) == '1'
    
    # Lógica para plano (que pode não existir em 2019)
    if col_plano and col_plano in df.columns and df[col_plano].notna().any():
        is_plano = df[col_plano].astype(str).str.replace(r'\.0$', '', regex=True) == '1'
    else:
        # Se não foi SUS nem Pago, assumimos Plano/Convênio/Outro (residual)
        is_plano = (~is_sus) & (~is_pago)

    condicoes = [is_sus, is_plano, is_pago]
    escolhas = ['SUS', 'Plano', 'Pagou']
    return np.select(condicoes, escolhas, default='Outros')

def calcular_decil_seguro(df):
    """
    Calcula decis forçando divisão em 10 grupos mesmo com valores repetidos (ex: muitos zeros),
    usando rank(method='first').
    """
    # Preenche NaNs com 0 para o cálculo
    series = df["renda_domiciliar_pc"].fillna(0)
    # O rank 'first' desempata pela ordem de aparição, permitindo que o qcut funcione
    return pd.qcut(series.rank(method='first'), 10, labels=False) + 1


In [ ]:
# ==============================================================================
# 2. REGISTRO DE VARIÁVEIS (QUADRO 1 DO PAPER)
# ==============================================================================

# --- 2.1 Variáveis Explicativas ---

# Faixa Etária (Categorias do paper)
register_derived_variable(
    name="faixa_etaria_paper",
    description="Faixas etárias: 25-34, 35-44, 45-54, 55-64, 65+",
    depends_on=["idade"],
    func=lambda df: pd.cut(
        df["idade"], 
        bins=[24, 34, 44, 54, 64, 150], 
        labels=['25-34', '35-44', '45-54', '55-64', '65+']
    )
)

# Raça (Dummy Branca)
register_derived_variable(
    name="dummy_branca",
    description="1 se Branca (código 1), 0 caso contrário",
    depends_on=["cor_raca"],
    func=lambda df: (df["cor_raca"].astype(str).str.replace(r'\.0$', '', regex=True) == '1').astype(int)
)

# Estado Civil (Casada)
register_derived_variable(
    name="dummy_casada",
    description="1 se casada ou vive com companheiro",
    depends_on=["estado_civil", "vive_com_companheiro"],
    # Paper considera união estável. C011=1 (Casado) ou vive_com_companheiro=1
    func=lambda df: (
        (df["estado_civil"].astype(str).str.contains('1')) | 
        (df["vive_com_companheiro"].astype(str).str.contains('1'))
    ).astype(int)
)

# Filhos (Dummy)
register_derived_variable(
    name="dummy_filhos",
    description="1 se tem filhos nascidos vivos > 0",
    depends_on=["tem_filhos_nascidos_vivos"],
    func=lambda df: (df["tem_filhos_nascidos_vivos"].fillna(0) > 0).astype(int)
)

# Escolaridade (Anos e Anos^2)
register_derived_variable(
    name="anos_estudo_estimado",
    description="Anos de estudo (proxy via escolaridade_nivel)",
    depends_on=["escolaridade_nivel"],
    func=calcular_anos_estudo_proxy
)

register_derived_variable(
    name="anos_estudo_sq",
    description="Anos de estudo ao quadrado (para modelo não-linear)",
    depends_on=["anos_estudo_estimado"],
    func=lambda df: df["anos_estudo_estimado"] ** 2
)

# Trabalho
register_derived_variable(
    name="dummy_trabalha",
    description="1 se ocupada na semana de referência (código 1)",
    depends_on=["trabalha"],
    func=lambda df: (df["trabalha"].astype(str).str.replace(r'\.0$', '', regex=True) == '1').astype(int)
)

# Renda (Decis)
register_derived_variable(
    name="decil_renda",
    description="Decil de renda domiciliar per capita (1 a 10)",
    depends_on=["renda_domiciliar_pc"],
    func=calcular_decil_seguro
)

# Região (Macro)
register_derived_variable(
    name="regiao_macro",
    description="Norte, Nordeste, Sudeste, Sul, Centro-Oeste (via UF)",
    depends_on=["uf"],
    func=lambda df: df["uf"].astype(str).str[0].map({
        '1': 'Norte', '2': 'Nordeste', '3': 'Sudeste', '4': 'Sul', '5': 'Centro-Oeste'
    })
)

# Situação Censitária (Urbano)
register_derived_variable(
    name="dummy_urbano",
    description="1 se Urbano (código 1)",
    depends_on=["situacao_censitaria"],
    func=lambda df: (df["situacao_censitaria"].astype(str).str.replace(r'\.0$', '', regex=True) == '1').astype(int)
)

# --- 2.2 Variáveis de Desfecho (Targets) ---

# Fez Preventivo (Binário)
register_derived_variable(
    name="fez_preventivo_bin",
    description="1 se fez preventivo (resposta != 5/Nunca), 0 se Nunca",
    depends_on=["preventivo_quando"],
    func=lambda df: np.where(
        df["preventivo_quando"].astype(str).str.replace(r'\.0$', '', regex=True) == '5', 0, 1
    )
)

# Fez Mamografia (Binário)
register_derived_variable(
    name="fez_mamografia_bin",
    description="1 se fez mamografia (1,2,3 ou 4), 0 caso contrário",
    depends_on=["mamografia_quando"],
    func=lambda df: np.where(
        df["mamografia_quando"].astype(str).str.replace(r'\.0$', '', regex=True).isin(['1','2','3','4']), 1, 0
    )
)

# Cobertura Preventivo (Multinomial)
register_derived_variable(
    name="cobertura_preventivo",
    description="SUS, Plano, Pagou ou Outros",
    depends_on=["preventivo_sus", "preventivo_pago", "preventivo_plano"],
    func=lambda df: classificar_cobertura_explicita(
        df, 
        col_sus="preventivo_sus", 
        col_pago="preventivo_pago", # Masculino no mapping
        col_plano="preventivo_plano"
    )
)


# Cobertura Mamografia (Multinomial)
register_derived_variable(
    name="cobertura_mamografia",
    description="SUS, Plano, Pagou ou Outros",
    depends_on=["mamografia_sus", "mamografia_paga", "mamografia_plano"],
    func=lambda df: classificar_cobertura_explicita(
        df, 
        col_sus="mamografia_sus", 
        col_pago="mamografia_paga", # Feminino no mapping
        col_plano="mamografia_plano"
    )
)

In [ ]:
# ==============================================================================
# 3. POPULANDO O CACHE (PREPARAÇÃO)
# ==============================================================================

# Definimos as variáveis que usaremos em TODAS as tabelas daqui pra frente.
# Ao chamar get_dataframe agora, seu sistema vai baixar do BQ (se necessário),
# calcular as derivadas acima e salvar no SQLite.
vars_base = [
    "sexo", "idade", "cor_raca", "estado_civil", "vive_com_companheiro",
    "tem_filhos_nascidos_vivos", "escolaridade_nivel", "trabalha",
    "renda_domiciliar_pc", "uf", "situacao_censitaria",
    "preventivo_quando", "preventivo_sus", "preventivo_pago", "preventivo_plano",
    "mamografia_quando", "mamografia_sus", "mamografia_paga", "mamografia_plano",
    "peso_amostral"
]

vars_derivadas = [
    "faixa_etaria_paper", "dummy_branca", "dummy_casada", "dummy_filhos",
    "anos_estudo_estimado", "anos_estudo_sq", "dummy_trabalha", "decil_renda",
    "regiao_macro", "dummy_urbano",
    "fez_preventivo_bin", "cobertura_preventivo",
    "fez_mamografia_bin", "cobertura_mamografia"
]

# Combina as listas (set para evitar duplicatas)
vars_totais = list(set(vars_base) | set(vars_derivadas))

print("Baixando dados e calculando derivadas...")

# As dependências estão na lista, então as derivadas serão calculadas.
df_pns = get_dataframe(
    variables=vars_totais,
    sources=["2013", "2019"],
    filters={"sexo": "2", "idade": {"operador": ">=", "valor": 25}}
)

print(f"Sucesso! DataFrame carregado com {len(df_pns)} linhas.")
print("Colunas disponíveis:", df_pns.columns.tolist())

print("Verificação de colunas críticas:")
print(df_pns[['decil_renda', 'cobertura_mamografia', 'cobertura_preventivo']].head())

In [ ]:
# Recarregando para forçar o cálculo da derivada em cadeia (anos_estudo_sq)
# que falhou na primeira passada.

print("Consolidando variáveis faltantes (anos_estudo_sq)...")

df_pns = get_dataframe(
    variables=vars_totais, # Mesma lista de antes
    sources=["2013", "2019"],
    filters={"sexo": "2", "idade": {"operador": ">=", "valor": 25}}
)

# Verificação Final de Integridade
cols_essenciais = [
    'anos_estudo_sq', 
    'anos_estudo_estimado', 
    'decil_renda', 
    'faixa_etaria_paper'
]

missing = [c for c in cols_essenciais if c not in df_pns.columns]

if missing:
    print(f"ATENÇÃO: Ainda faltam colunas: {missing}")
else:
    print("Sucesso Total! Todas as variáveis calculadas.")
    print("\nVisualização dos dados tratados:")
    print(df_pns[cols_essenciais].head())
    print(df_pns[cols_essenciais].describe())